# Day 64: MNIST — First Real Neural Network Project
## Build, Train, and Evaluate a Complete DL Pipeline

---

# PART 1: THEORY

## 1. The MNIST Dataset — "Hello World" of DL

- **70,000 images** of handwritten digits (0-9)
- Each image = 28x28 pixels = 784 features
- **Training:** 60,000 images
- **Testing:** 10,000 images
- 10 classes (digits 0 through 9)
- Balanced dataset (~6,000 per digit)

**Why MNIST?** Small enough to train on CPU in minutes. Complex enough to demonstrate real DL concepts. The perfect first dataset.

## 2. From Images to Neural Network Input

An image is a 2D grid of pixels. A neural network expects a 1D vector.

```
28x28 image = 784 pixel values -> Flatten -> 784 input neurons
```

**Preprocessing steps:**
1. Normalize pixel values to [0, 1] (divide by 255)
2. Flatten 28x28 to 784 (for Dense layers)
3. One-hot encode labels (0 -> [1,0,0,...], 5 -> [0,0,0,0,0,1,0,...])

## 3. Model Architecture Design

For MNIST, a simple Dense network works well:
- Input: 784 neurons
- Hidden 1: 128 neurons (ReLU)
- Hidden 2: 64 neurons (ReLU)
- Output: 10 neurons (Softmax) — one per digit

## 4. Evaluating Classification Models

- **Accuracy:** % of correct predictions
- **Confusion Matrix:** Shows which digits get confused
- **Classification Report:** Precision, Recall, F1 per class

---

# PART 2: PRACTICAL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import seaborn as sns


## 5. Load and Explore MNIST

In [ ]:
# Load MNIST (auto-downloads!)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"Image size: {X_train.shape[1]}x{X_train.shape[2]}")
print(f"Pixel range: [{X_train.min()}, {X_train.max()}]")
print(f"Classes: {len(np.unique(y_train))} digits")
print(f"\nClass distribution (train):")
for i in range(10):
    print(f"  Digit {i}: {(y_train == i).sum()} images")


In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(y_train == i)[0][0]  # First occurrence of each digit
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(f'Digit: {y_train[idx]}', fontsize=14, fontweight='bold')
    ax.axis('off')
plt.suptitle('MNIST — One Sample Per Digit', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Show grid of random digits
np.random.seed(42)
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for ax in axes.flatten():
    idx = np.random.randint(0, len(X_train))
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Random MNIST Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Preprocess Data

In [ ]:
# Normalize pixel values to [0, 1]
X_train_n = X_train.astype('float32') / 255.0
X_test_n = X_test.astype('float32') / 255.0

# Flatten 28x28 -> 784 for Dense layers
X_train_f = X_train_n.reshape(-1, 784)
X_test_f = X_test_n.reshape(-1, 784)

# One-hot encode labels
y_train_oh = keras.utils.to_categorical(y_train, 10)
y_test_oh = keras.utils.to_categorical(y_test, 10)

print(f"X_train: {X_train_f.shape} (flattened)")
print(f"X_test:  {X_test_f.shape} (flattened)")
print(f"y_train: {y_train_oh.shape} (one-hot)")
print(f"\nExample: Label {y_train[0]} -> One-hot {y_train_oh[0]}")


## 7. Build and Train the Model

In [ ]:
# Build the model
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dropout(0.2),  # Light regularization
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:
# Train with early stopping
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(X_train_f, y_train_oh, epochs=30, batch_size=32,
                    validation_split=0.1, callbacks=[early_stop], verbose=1)


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 8. Evaluate the Model

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test_f, y_test_oh, verbose=0)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test Loss: {test_loss:.4f}")


In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

y_pred_probs = model.predict(X_test_f, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Digit', fontsize=12)
plt.ylabel('True Digit', fontsize=12)
plt.title('Confusion Matrix — MNIST', fontsize=14, fontweight='bold')
plt.show()

# Most confused digit pairs
errors = []
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 5:
            errors.append((i, j, cm[i, j]))
errors.sort(key=lambda x: -x[2])
print("Most confused digit pairs:")
for true, pred, count in errors[:5]:
    print(f"  True {true} predicted as {pred}: {count} times")

# Classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))


## 9. Visualize Predictions

In [ ]:
# Show correct and incorrect predictions
y_probs = model.predict(X_test_f, verbose=0)
y_pred = np.argmax(y_probs, axis=1)
y_conf = np.max(y_probs, axis=1)

# Find correct and incorrect indices
correct = np.where(y_pred == y_test)[0]
incorrect = np.where(y_pred != y_test)[0]

print(f"Correct predictions: {len(correct)}")
print(f"Incorrect predictions: {len(incorrect)}")

# Show 6 best and 6 worst predictions
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i, (idx, ax) in enumerate(zip(np.random.choice(correct, 6, replace=False), axes[0])):
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f'Pred: {y_pred[idx]} (True: {y_test[idx]}), Conf: {y_conf[idx]:.1%}', color='green')
    ax.axis('off')

for i, (idx, ax) in enumerate(zip(np.random.choice(incorrect, 6, replace=False), axes[1])):
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f'Pred: {y_pred[idx]} (True: {y_test[idx]}), Conf: {y_conf[idx]:.1%}', color='red')
    ax.axis('off')

axes[0, 0].set_ylabel('CORRECT', fontsize=14, fontweight='bold', labelpad=40)
axes[1, 0].set_ylabel('WRONG', fontsize=14, fontweight='bold', labelpad=40)
plt.suptitle('MNIST Predictions — Best vs Worst', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Try deeper/wider architectures
# Try: [256, 128, 64], [512, 256, 128, 64], etc.
# Which gives the best test accuracy?
archs = {
    'Baseline [128, 64]': [128, 64],
    'Wide [256, 128]': [256, 128],
    'Deep [256, 128, 64]': [256, 128, 64],
    'Very Deep [512, 256, 128, 64]': [512, 256, 128, 64],
}
for name, units in archs.items():
    m = keras.Sequential(
        [layers.Dense(u, activation='relu', input_shape=(784,) if i==0 else None) for i, u in enumerate(units)] +
        [layers.Dense(10, activation='softmax')]
    )
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train_f, y_train_oh, epochs=5, validation_split=0.1, verbose=0)
    _, acc = m.evaluate(X_test_f, y_test_oh, verbose=0)
    print(f"{name:35s} -> Test Acc: {acc:.4f} | Params: {m.count_params():,}")


In [ ]:
# Exercise 2: Test on your own handwriting!
# Draw a digit (28x28), save it, and test the model
# Or use this code to select a test image and see the prediction
idx = np.random.randint(0, len(X_test))
sample = X_test[idx]
sample_n = sample.astype('float32') / 255.0
sample_f = sample_n.reshape(1, 784)

prediction = model.predict(sample_f, verbose=0)
pred_label = np.argmax(prediction)
true_label = y_test[idx]

plt.imshow(sample, cmap='gray')
plt.title(f'True: {true_label}, Predicted: {pred_label}', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()


## Key Takeaways

- MNIST = the perfect first DL project (small, fast, meaningful)
- **Normalize** pixel values to [0, 1] — mandatory for neural networks
- **One-hot encode** labels for multi-class classification
- **Softmax** output gives probability distribution across 10 digits
- A simple 2-layer network achieves **~97% accuracy** in minutes
- **Confusion matrix** reveals which digits the model confuses
- Model confidence (probability) tells you how sure the prediction is

**Tomorrow:** CNNs — the architecture that revolutionized computer vision!